# Análise Exploratória (EDA) — Inteligência Macroeconômica BCB/SGS

**Recorte:** janeiro de 2020 a dezembro de 2023. **Fonte:** SGS (Banco Central do Brasil), séries 432, 433, 13522, 1, 24364.

O notebook apresenta estatísticas descritivas, limpeza, análise de ausências e duplicados, outliers, dispersão, distribuição e correlações. As bases devem ser previamente preparadas executando `python -m src.etl` na raiz do projeto. **Nenhum número é simulado.**

> Importante: resultados de associação não estabelecem causalidade econômica.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display, Markdown

ROOT = Path.cwd()
if not (ROOT / "data" / "processed").exists():
    ROOT = ROOT.parent
BASE = ROOT / "data" / "processed"
mensal = pd.read_csv(BASE / "macroeconomia_mensal.csv", parse_dates=["data"])
ptax = pd.read_csv(BASE / "ptax_diaria.csv", parse_dates=["data"])
mensal = mensal.sort_values("data").reset_index(drop=True)
ptax = ptax.sort_values("data").reset_index(drop=True)
print(f"Observações mensais: {len(mensal)} | PTAX diária: {len(ptax)}")
display(mensal.head(5))

## 1. Inspeção dos tipos, nulos e duplicidades

As cinco séries possuem periodicidades diferentes. A harmonização mensal utiliza o último registro da Selic, a média mensal da PTAX e os valores mensais originais de IPCA e IBC-Br. A ausência de publicação não recebe preenchimento arbitrário (exceto manutenção da meta Selic).

In [ ]:
display(pd.DataFrame({
    "tipo": mensal.dtypes.astype(str),
    "nulos": mensal.isna().sum(),
    "nulos_%": (100 * mensal.isna().mean()).round(2),
}))
print("Linhas duplicadas:", int(mensal.duplicated().sum()))
print("Meses duplicados:", int(mensal["data"].duplicated().sum()))
print("Extremos temporais:", mensal["data"].min(), "até", mensal["data"].max())
assert not mensal["data"].duplicated().any(), "Meses duplicados detectados"
assert mensal["data"].min().year == 2020
assert mensal["data"].max().year == 2023

## 2. Estatísticas descritivas por indicador

In [ ]:
estatisticas = mensal.drop(columns=["data"]).describe(percentiles=[.25, .5, .75]).T
estatisticas["assimetria"] = mensal.drop(columns=["data"]).skew(numeric_only=True)
display(estatisticas.round(3))

## 3. Outliers com intervalo interquartil (IQR)

Pontos fora de Q1−1,5×IQR ou Q3+1,5×IQR são sinalizados, mas **não excluídos** automaticamente: choques macroeconômicos podem ser eventos legítimos e informativos.

In [ ]:
campos = ["selic", "ipca_mensal", "ipca_12m", "ptax", "ibc_br", "juro_real"]
linhas = []
for campo in campos:
    s = mensal[campo].dropna()
    if s.empty:
        continue
    q1, q3 = s.quantile([.25, .75])
    iqr = q3 - q1
    inferior, superior = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    linhas.append({
        "série": campo, "Q1": q1, "Q3": q3,
        "limite_inferior": inferior, "limite_superior": superior,
        "outliers_IQR": int(((s < inferior) | (s > superior)).sum()),
    })
display(pd.DataFrame(linhas).round(3))
fig = px.box(
    mensal, y=["selic", "ipca_12m", "juro_real"], points="outliers",
    title="Distribuição de taxas: mediana, quartis e valores extremos",
    labels={"value": "Percentual (%)", "variable": "Indicador"},
)
display(fig)

## 4. Dispersão, possível associação e distribuição dos retornos

A dispersão de Selic × IBC-Br é apenas um retrato simultâneo. Os gráficos não têm interpretação de impacto causal. Abaixo também se observa a distribuição mensal do IPCA, com eixo e unidade explícitos.

In [ ]:
fig = px.scatter(
    mensal, x="selic", y="ibc_br", color=mensal["data"].dt.year.astype(str),
    hover_data=["data"], title="Selic x IBC-Br (níveis mensais)",
    labels={"selic": "Selic meta (% a.a.)", "ibc_br": "IBC-Br dessazonalizado",
            "color": "Ano"},
    trendline=None,
)
display(fig)
fig = px.histogram(
    mensal, x="ipca_mensal", nbins=18, marginal="box",
    title="Distribuição do IPCA mensal (2020–2023)",
    labels={"ipca_mensal": "IPCA no mês (%)", "count": "Frequência"},
)
display(fig)

## 5. Correlação linear de Pearson e cautela analítica

Amostra mensal alinhada. Série em nível e autocorrelacionada pode apresentar coeficiente alto devido à tendência comum. O juro real também depende algebricamente de Selic e IPCA, portanto suas correlações não são independentes.

In [ ]:
variaveis = ["selic", "ipca_12m", "ptax", "ibc_br", "juro_real"]
corr = mensal[variaveis].corr(method="pearson", min_periods=6)
display(corr.round(3))
fig = px.imshow(
    corr, zmin=-1, zmax=1, color_continuous_scale="RdBu_r",
    text_auto=".2f", title="Correlação de Pearson (níveis mensais)",
)
display(fig)
pares = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack()
if len(pares):
    maior = pares.abs().idxmax()
    display(Markdown(
        f"**Maior correlação absoluta:** {maior[0]} × {maior[1]}: "
        f"r = {pares.loc[maior]:+.3f}. Isso **não** implica causalidade."
    ))

## 6. Interpretação e limites

- A análise é **retrospectiva** (2020–2023), não apresenta valores vigentes em 2026.
- Frequências foram padronizadas para meses; PTAX é a média das observações **diárias disponíveis**.
- Meta Selic é informada em % ao ano; IPCA 12m é inflação retrospectiva de 12 meses.
- A aproximação do juro real ex-post usa `((1 + Selic/100)/(1 + IPCA_12m/100) - 1)*100`.
- Valores extremos são mantidos como possíveis choques legítimos.
- Séries em níveis exigiriam análise de estacionariedade, defasagens e controles para inferência econômica causal.